# Bounce detector training on Colab (LSTM)

Trains a small windowed-sequence LSTM to classify each bounce candidate as a real ground bounce vs. a racket contact/noise, using the RAW (x, y) ball trajectory shape instead of hand-picked features - see `scripts/train_bounce_lstm.py`'s module docstring for why, and how this complements (not replaces) `weights/bounce_classifier.pkl`'s RandomForest.

**Before running:** upload `bounce_lstm_training_data.zip` (built locally - just the labeled candidate CSVs, tiny: `outputs/*_labeled.csv` from `scripts/extract_bounce_candidates.py` + the reference/TrackNet importers, all sharing the same `--window` value) to your Google Drive at:

`My Drive/bounce_lstm_training_data.zip`

Then: Runtime -> Change runtime type -> T4 GPU (optional - this model is small enough that CPU works fine too, GPU just makes iterating faster), and run the cells in order.

**Before labeling more data:** favor far-court candidates over near-court ones when filling in `outputs/bounce_candidates_labeled.csv`'s `label` column. A real bounce's pixel dip shrinks a lot with distance from the camera (perspective), so if the labeled set skews near-court, the model will tend to learn "small dip = not a bounce" and miss exactly the far-court bounces this was meant to catch - the same lesson [the reference project](https://github.com/s-ganguli/AI-Tennis-Ball-Bounce-Detection) applied by curating a 1:2 near:far ratio for its own object-detection training set.

In [ ]:
!pip install -q tensorflow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Unzip the labeled-CSV bundle from Drive
!mkdir -p /content/bounce_data
!unzip -q "/content/drive/MyDrive/bounce_lstm_training_data.zip" -d /content/bounce_data
!find /content/bounce_data -name "*.csv"

In [ ]:
# Clone the repo (public GitHub) at the working branch
!git clone -b claude/tennis-ball-yolo-tracking-p8ntwh --depth 1 https://github.com/will-wang1/tennis-tracking-yolo-v1.git /content/repo
%cd /content/repo
!pip install -q -r requirements.txt

In [ ]:
import glob

label_csvs = glob.glob('/content/bounce_data/**/*.csv', recursive=True)
print(f"{len(label_csvs)} labeled CSV(s) found:")
for path in label_csvs:
    print(' ', path)

In [ ]:
!mkdir -p /content/drive/MyDrive/tennis_colab/bounce_lstm_runs
!python scripts/train_bounce_lstm.py \
    --labels {' '.join(label_csvs)} \
    --out /content/drive/MyDrive/tennis_colab/bounce_lstm_runs/bounce_lstm.keras \
    --epochs 150 \
    --cv-folds 5

## After training

The trained model (`bounce_lstm.keras` + its companion `bounce_lstm.keras.json`, both needed) is at `My Drive/tennis_colab/bounce_lstm_runs/` - no extra download step needed inside Colab, it's already there. On your local machine:

1. Download both `bounce_lstm.keras` and `bounce_lstm.keras.json` from that Drive folder into `weights/` (same directory, filenames unchanged - main.py looks for the `.json` right next to the model).
2. Try it out layered on top of the existing geometric bounce detection as a contact-vs-bounce veto:
   ```
   python main.py --input your_video.mp4 --output outputs/test.mp4 \
       --pose --bounce --bounce-classifier weights/bounce_lstm.keras \
       --speed --sidebar --show-court
   ```
3. `pip install tensorflow` locally if you want to run this path outside Colab too (it's not in `requirements.txt` - only needed if you actually use `--bounce-classifier` pointed at a `.keras`/`.h5` file).
4. `python -m pytest tests/ -q` to sanity check nothing broke (the LSTM-specific tests are skipped automatically if `tensorflow` isn't installed).